# Game Success Classifier
## Can we predict whether a video game will be critically acclaimed?

**IOD Data Science & AI — Mini Project 03**
**Student:** Sean C
**GitHub:** [brighthikaru/game-success-classifier](https://github.com/brighthikaru/game-success-classifier)
**Data Source:** [RAWG Video Games Database API](https://rawg.io/apidocs) — collected June 2026
**Date:** July 2026

## Business Understanding

The video game industry generates over $180 billion in revenue annually — larger than both the film and music industries combined. With thousands of games released every year across dozens of platforms, understanding what separates a critically acclaimed game from one that does not land has real commercial value for developers, publishers, and investors.

A strong Metacritic score drives visibility on storefronts, influences purchasing decisions, and can determine the commercial fate of a studio. But what actually predicts critical success before or at release?

This project uses data collected directly from the RAWG Video Games Database API — one of the largest open gaming databases in the world — to explore this question through machine learning.

**Business question:** Can we predict whether a video game will be critically acclaimed based on its characteristics at release?

We define a Hit as a game with a Metacritic score of 75 or above, representing broadly positive critical reception. Anything below 75 is classified as Not Hit — not a commercial failure necessarily, but not critically acclaimed.

| Class | Metacritic Score | Meaning |
|---|---|---|
| Hit | >= 75 | Broadly positive critical reception |
| Not Hit | < 75 | Mixed or negative critical reception |

## Project Structure (CRISP-DM)

| Phase | Section |
|---|---|
| Business Understanding | Above |
| Data Understanding | Section 1 — Load and Inspect |
| Data Preparation | Section 2 — Clean and Engineer Features |
| Exploratory Analysis | Section 3 — EDA |
| Modelling | Section 4 — Feature Matrix and Section 5 — Models |
| Evaluation | Section 6 — Comparison and Metrics |
| Tuning | Section 7 — Hyperparameter Optimisation |
| Insights | Section 8 — Feature Importance |
| Conclusion | Section 9 |

## Reusable Functions

A key design goal of this notebook is software engineering thinking — wrapping repeated logic into clean, reusable functions with clear inputs, outputs, and docstrings. Each function does one thing, takes clear inputs, and returns clear outputs.

| Function | Purpose |
|---|---|
| `load_and_inspect()` | Load any CSV and print a full data quality report |
| `engineer_features()` | Parse and transform raw columns into model-ready features |
| `plot_class_distribution()` | Plot target variable balance for any binary classification task |
| `build_feature_matrix()` | Encode and scale features, return X and y ready for modelling |
| `train_and_evaluate()` | Train any model and return a consistent metrics dictionary |
| `compare_models()` | Build a styled summary table and bar chart across all models |
| `run_grid_search()` | Run GridSearchCV with consistent settings and reporting |
| `plot_confusion_matrix()` | Plot a styled confusion matrix for any fitted model |
| `plot_feature_importance()` | Extract and plot feature importance for linear and tree models |


## Imports and Configuration

In [4]:
# Standard library
import re
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

# Scikit-learn: models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Scikit-learn: evaluation
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc as sklearn_auc,
)

# Statsmodels — for statistical significance testing
import statsmodels.api as sm

warnings.filterwarnings("ignore")

# Global settings — random_state kept consistent throughout (lesson from MP02)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA_PATH    = Path("../data/rawg_games.csv")

# Colour palette — consistent across all charts (SWD principles)
BLUE   = "#3B6EA5"
CORAL  = "#C45C3A"
GREEN  = "#2E7D5E"
GOLD   = "#E6A817"
GREY   = "#AAAAAA"
PURPLE = "#7B5EA7"
PALETTE = [BLUE, CORAL, GREEN, GOLD, GREY, PURPLE]

plt.rcParams.update({
    "figure.dpi":        150,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
})



## Section 1 — Data Loading and Inspection

We load the dataset collected from the RAWG Video Games Database API and run a full data quality inspection before modifying anything.

The data was collected using a custom Python script (`src/collect_rawg_data.py`) that called the RAWG API, filtered to games with a Metacritic score, and saved results incrementally to CSV. The collection script is part of the project repository.

**Reusable function: `load_and_inspect()`**

Loads any CSV file and immediately reports shape, column types, missing values, and duplicate rows. Wrapping this as a function means we can run the same quality check on any dataset with a single call.


In [ ]:
def load_and_inspect(filepath: Path) -> pd.DataFrame:
    """
    Load a CSV file and print a full data quality report.

    Parameters
    ----------
    filepath : Path
        Path to the CSV file to load.

    Returns
    -------
    pd.DataFrame
        The loaded dataframe.
    """
    df = pd.read_csv(filepath)

    print(f"Dataset : {filepath.name}")
    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")
    print(f"Duplicates: {df.duplicated().sum():,}")
    print()
    print(f"  {'Column':<22} {'Type':<10} {'Nulls':>8}  {'Unique':>8}")
    print(f"  {'-'*52}")
    for col in df.columns:
        nulls  = df[col].isna().sum()
        unique = df[col].nunique()
        dtype  = str(df[col].dtype)
        flag   = "  <-- missing values" if nulls > 0 else ""
        print(f"  {col:<22} {dtype:<10} {nulls:>8,}  {unique:>8,}{flag}")

    return df


In [ ]:
# Load the dataset
df_raw = load_and_inspect(DATA_PATH)


In [ ]:
# Preview the first few rows
df_raw.head(3)


In [ ]:
# Sample key columns to understand the raw format
# genres, platforms, and tags are pipe-separated strings — not yet usable as features
print("Sample genres:")
print(df_raw['genres'].dropna().head(5).to_string())
print()
print("Sample platforms:")
print(df_raw['platforms'].dropna().head(5).to_string())
print()
print("Sample tags:")
print(df_raw['tags'].dropna().head(3).to_string())


**Observations:**

*(Fill in after running — what do you notice? What needs cleaning?)*

-
-
-


## Section 2 — Data Cleaning and Feature Engineering

Raw data from APIs is rarely model-ready. This section transforms the raw RAWG data into a clean feature matrix suitable for machine learning.

**Cleaning tasks:**

1. Parse pipe-separated columns — genres, platforms, and tags are stored as "Action|RPG|Adventure" strings. We extract the primary value and engineer count features from each.
2. Clean tags — the tags column contains mixed English and Russian text from RAWG's international user base. We filter to English-only tags.
3. Handle missing ESRB ratings — 33% of games have no ESRB rating recorded. We encode these as "Not Rated" rather than dropping the rows.
4. Log-transform playtime — high playtime is genuine signal in gaming (a player logging 500 hours in an RPG is expected, not an outlier). We apply a log transformation to compress the right-skewed distribution without discarding any data.
5. Engineer release features — we extract release year and create an `is_holiday_release` flag for November and December releases, which historically perform differently.
6. Build the target variable — `hit = 1` if Metacritic >= 75, else `hit = 0`.

**Reusable function: `engineer_features()`**

Encapsulating all feature engineering in one function ensures the same transformations are applied consistently, and makes the logic easy to reuse on future gaming datasets.


In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean and engineer features from the raw RAWG dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe as loaded from rawg_games.csv.

    Returns
    -------
    pd.DataFrame
        Cleaned dataframe with engineered features and target variable added.

    Features engineered
    -------------------
    primary_genre      : First (most prominent) genre listed
    n_genres           : Number of genres assigned to the game
    primary_platform   : First platform listed
    n_platforms        : Number of platforms (multi-platform suggests larger budget)
    primary_tag        : First English-language tag
    n_tags             : Number of English tags
    esrb_clean         : ESRB rating with nulls filled as 'Not Rated'
    playtime_log       : Log-transformed playtime — preserves high-engagement signal
    is_holiday_release : 1 if released in November or December
    release_year       : Year of release
    hit                : Target variable — 1 if metacritic >= 75, else 0
    """
    df = df.copy()

    # Parse pipe-separated genre column
    df['genres'] = df['genres'].fillna("")
    df['primary_genre'] = df['genres'].apply(
        lambda x: x.split("|")[0].strip() if x else "Unknown"
    )
    df['n_genres'] = df['genres'].apply(
        lambda x: len(x.split("|")) if x else 0
    )

    # Parse pipe-separated platform column
    df['platforms'] = df['platforms'].fillna("")
    df['primary_platform'] = df['platforms'].apply(
        lambda x: x.split("|")[0].strip() if x else "Unknown"
    )
    df['n_platforms'] = df['platforms'].apply(
        lambda x: len(x.split("|")) if x else 0
    )

    # Parse tags — keep English only (filter out Cyrillic characters)
    def extract_english_tags(tag_string):
        """Return only tags that contain no Cyrillic characters."""
        if not isinstance(tag_string, str) or tag_string == "":
            return []
        tags = tag_string.split("|")
        return [t.strip() for t in tags if not re.search(r'[а-яА-ЯёЁ]', t)]

    df['tags_clean']  = df['tags'].apply(extract_english_tags)
    df['primary_tag'] = df['tags_clean'].apply(lambda x: x[0] if x else "Unknown")
    df['n_tags']      = df['tags_clean'].apply(len)

    # ESRB rating — fill nulls as 'Not Rated'
    df['esrb_clean'] = df['esrb_rating'].fillna("Not Rated")

    # Log-transform playtime — high playtime is genuine signal (e.g. 500hrs in Civilisation
    # is normal and meaningful). Capping would destroy that signal. Instead we apply
    # log1p (log(1 + x)) which compresses the long tail while preserving the ranking.
    # log1p handles zeros safely — log1p(0) = 0.
    df['playtime_log'] = np.log1p(df['playtime'])

    # Holiday release flag — November and December releases
    df['release_month']      = pd.to_numeric(df['release_month'], errors='coerce').fillna(0).astype(int)
    df['is_holiday_release'] = df['release_month'].isin([11, 12]).astype(int)

    # Release year — clean and fill missing with median
    df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
    df['release_year'] = df['release_year'].fillna(df['release_year'].median()).astype(int)

    # Target variable
    df['hit'] = (df['metacritic'] >= 75).astype(int)

    return df


In [ ]:
# Apply feature engineering
df = engineer_features(df_raw)

print(f"Shape after engineering: {df.shape}")
print()
print("New columns added:")
new_cols = [
    'primary_genre', 'n_genres', 'primary_platform', 'n_platforms',
    'primary_tag', 'n_tags', 'esrb_clean', 'playtime_log',
    'is_holiday_release', 'release_year', 'hit',
]
for col in new_cols:
    print(f"  {col:<25} {str(df[col].dtype):<10}  nulls: {df[col].isna().sum()}")


In [ ]:
# Target variable distribution
print("Target variable — Hit vs Not Hit:")
print(df['hit'].value_counts().rename({1: 'Hit (>=75)', 0: 'Not Hit (<75)'}).to_string())
print(f"\nClass balance: {df['hit'].mean()*100:.1f}% Hit")


**Feature engineering summary:**

*(Fill in after running)*

-
-
-


## Section 3 — Exploratory Data Analysis

Before building models we visualise the data to understand distributions, class balance, and which features look most predictive of critical success. Every chart is followed by an insight note explaining what it means for modelling.

**Reusable function: `plot_class_distribution()`**


In [ ]:
def plot_class_distribution(y: pd.Series,
                            labels: dict = None,
                            title: str = "Target Variable Distribution",
                            colours: list = None) -> None:
    """
    Plot the distribution of a binary target variable as a bar chart and pie chart.

    Parameters
    ----------
    y : pd.Series
        Binary target variable (0/1 values).
    labels : dict, optional
        Mapping of {0: 'Label A', 1: 'Label B'} for display.
    title : str
        Chart title.
    colours : list, optional
        Two colours for the bars. Defaults to CORAL and BLUE.
    """
    if colours is None:
        colours = [CORAL, BLUE]
    if labels is None:
        labels = {0: "Class 0", 1: "Class 1"}

    counts = y.value_counts().sort_index().rename(labels)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    bars = axes[0].bar(counts.index, counts.values,
                       color=colours, edgecolor="white", width=0.5)
    for bar, val in zip(bars, counts.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 30,
                     f"{val:,}", ha="center", fontsize=10)
    axes[0].set_title(title, fontweight="bold")
    axes[0].set_ylabel("Count")

    axes[1].pie(counts.values, labels=counts.index,
                colors=colours, autopct="%1.1f%%",
                startangle=90, wedgeprops={"edgecolor": "white"})
    axes[1].set_title("Class balance", fontweight="bold")

    plt.tight_layout()
    plt.savefig("../outputs/class_distribution.png", bbox_inches="tight")
    plt.show()


In [ ]:
# 3.1 Target variable distribution
plot_class_distribution(
    df['hit'],
    labels={0: "Not Hit (<75)", 1: "Hit (>=75)"},
    title="Hit vs Not Hit — Metacritic threshold 75",
)


**Insight:** *(Fill in after running)*


In [ ]:
# 3.2 Metacritic score distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['metacritic'], bins=40, color=BLUE, edgecolor="white", alpha=0.85)
ax.axvline(75, color=CORAL, linestyle="--", linewidth=2, label="Hit threshold (75)")
ax.set_title("Metacritic score distribution", fontweight="bold")
ax.set_xlabel("Metacritic Score")
ax.set_ylabel("Number of games")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/metacritic_distribution.png", bbox_inches="tight")
plt.show()


**Insight:** *(Fill in after running)*


In [ ]:
# 3.3 Playtime distribution — log scale
# High playtime is natural and expected (e.g. RPGs, MMOs, strategy games)
# We use log scale on the x-axis to show the full range clearly
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Raw playtime distribution
axes[0].hist(df['playtime'], bins=50, color=BLUE, edgecolor="white", alpha=0.85)
axes[0].set_title("Playtime distribution (raw)", fontweight="bold")
axes[0].set_xlabel("Average playtime (hours)")
axes[0].set_ylabel("Number of games")

# Log-transformed distribution
axes[1].hist(df['playtime_log'], bins=50, color=GREEN, edgecolor="white", alpha=0.85)
axes[1].set_title("Playtime distribution (log-transformed)", fontweight="bold")
axes[1].set_xlabel("log(1 + playtime)")
axes[1].set_ylabel("Number of games")

plt.suptitle("Playtime before and after log transformation", fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/playtime_distribution.png", bbox_inches="tight")
plt.show()

print(f"Raw playtime   — mean: {df['playtime'].mean():.1f}h  median: {df['playtime'].median():.1f}h  max: {df['playtime'].max():.0f}h")
print(f"Log playtime   — mean: {df['playtime_log'].mean():.2f}  median: {df['playtime_log'].median():.2f}  max: {df['playtime_log'].max():.2f}")


**Insight:** The raw playtime distribution is heavily right-skewed — most games have low average playtime, but some titles (open-world RPGs, strategy games, MMOs) naturally attract hundreds of hours of play. This is not an outlier problem — it is genuine signal. We apply a log transformation to compress the scale without removing data, making it more suitable as a model feature while preserving the information that high-engagement games carry.


In [ ]:
# 3.3 Hit rate by genre
genre_hit = (df.groupby('primary_genre')['hit']
               .agg(['mean', 'count'])
               .rename(columns={'mean': 'hit_rate', 'count': 'total'})
               .query('total >= 30')
               .sort_values('hit_rate', ascending=True))

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(genre_hit.index, genre_hit['hit_rate'] * 100,
               color=BLUE, edgecolor="white")
for bar, val in zip(bars, genre_hit['hit_rate'] * 100):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9)
ax.axvline(df['hit'].mean() * 100, color=CORAL, linestyle="--",
           linewidth=1.5, label=f"Overall average ({df['hit'].mean()*100:.1f}%)")
ax.set_xlabel("Hit rate (%)")
ax.set_title("Hit rate by primary genre", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/hit_rate_by_genre.png", bbox_inches="tight")
plt.show()


**Insight:** *(Fill in after running)*


In [ ]:
# 3.4 Hit rate by ESRB rating
esrb_hit = (df.groupby('esrb_clean')['hit']
              .agg(['mean', 'count'])
              .rename(columns={'mean': 'hit_rate', 'count': 'total'})
              .sort_values('hit_rate', ascending=True))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(esrb_hit.index, esrb_hit['hit_rate'] * 100,
               color=GOLD, edgecolor="white")
for bar, val in zip(bars, esrb_hit['hit_rate'] * 100):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9)
ax.axvline(df['hit'].mean() * 100, color=CORAL, linestyle="--",
           linewidth=1.5, label="Overall average")
ax.set_xlabel("Hit rate (%)")
ax.set_title("Hit rate by ESRB rating", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/hit_rate_by_esrb.png", bbox_inches="tight")
plt.show()


**Insight:** *(Fill in after running)*


In [ ]:
# 3.5 Hit rate trend by release year
year_hit = (df[df['release_year'] >= 1995]
              .groupby('release_year')['hit']
              .agg(['mean', 'count'])
              .rename(columns={'mean': 'hit_rate', 'count': 'total'}))

fig, ax1 = plt.subplots(figsize=(11, 4))
ax2 = ax1.twinx()

ax1.plot(year_hit.index, year_hit['hit_rate'] * 100,
         color=BLUE, linewidth=2, marker='o', markersize=3, label="Hit rate %")
ax2.bar(year_hit.index, year_hit['total'], color=GREY, alpha=0.3, label="Games in dataset")
ax1.axhline(df['hit'].mean() * 100, color=CORAL, linestyle="--",
            linewidth=1.2, label="Overall average")

ax1.set_ylabel("Hit rate (%)", color=BLUE)
ax2.set_ylabel("Games in dataset", color=GREY)
ax1.set_xlabel("Release year")
ax1.set_title("Hit rate and game volume by release year", fontweight="bold")
ax1.legend(loc="upper left")
plt.tight_layout()
plt.savefig("../outputs/hit_rate_by_year.png", bbox_inches="tight")
plt.show()


**Insight:** *(Fill in after running)*


In [ ]:
# 3.6 Correlation heatmap — numeric features
numeric_cols = [
    'metacritic', 'rating', 'rating_count', 'playtime_log',
    'n_genres', 'n_platforms', 'n_tags', 'release_year', 'is_holiday_release',
]

corr     = df[numeric_cols].corr()
mask_arr = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, mask=mask_arr)
ax.set_title("Correlation heatmap — numeric features", fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/correlation_heatmap.png", bbox_inches="tight")
plt.show()


**Insight:** *(Fill in after running)*


## Section 4 — Feature Selection and Building the Feature Matrix

We select the features to include in modelling and encode categorical variables into numeric form.

**Reusable function: `build_feature_matrix()`**

Isolating feature selection and encoding in one function ensures the same preprocessing is applied consistently to training and test data.


In [ ]:
def build_feature_matrix(df: pd.DataFrame):
    """
    Select, encode, and return the feature matrix X and target y.

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned dataframe output from engineer_features().

    Returns
    -------
    X : pd.DataFrame
        Encoded feature matrix ready for modelling.
    y : pd.Series
        Binary target variable (hit = 1, not hit = 0).
    feature_names : list
        Names of all features in X.

    Features used
    -------------
    Numeric    : playtime_log, n_genres, n_platforms, n_tags,
                 release_year, is_holiday_release, rating_count
    Encoded    : primary_genre, esrb_clean (label encoded)

    Features excluded
    -----------------
    metacritic : This IS the target variable. Including it would be data leakage.
    rating     : RAWG community score — strongly correlated with metacritic and
                 not available as a pre-release feature in a real-world scenario.
    name, id   : Identifier columns, not predictive.
    """
    numeric_features     = [
        'playtime_log', 'n_genres', 'n_platforms', 'n_tags',
        'release_year', 'is_holiday_release', 'rating_count',
    ]
    categorical_features = ['primary_genre', 'esrb_clean']

    X = df[numeric_features].copy()

    for col in categorical_features:
        le    = LabelEncoder()
        X[col] = le.fit_transform(df[col].astype(str))

    y             = df['hit']
    feature_names = numeric_features + categorical_features

    print(f"Feature matrix shape: {X.shape}")
    print(f"\nFeatures ({len(feature_names)}):")
    for f in feature_names:
        print(f"  {f}")
    print(f"\nTarget balance: {y.mean()*100:.1f}% Hit")

    return X, y, feature_names


In [ ]:
# Build feature matrix
X, y, feature_names = build_feature_matrix(df)

# Train / test split — stratify=y keeps the same Hit/Not Hit ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")
print(f"\nClass balance in train : {y_train.mean()*100:.1f}% Hit")
print(f"Class balance in test  : {y_test.mean()*100:.1f}% Hit")


**Why we exclude metacritic and rating as features:**

`metacritic` is the target variable — including it as a feature would be perfect data leakage. The model would simply learn "if metacritic is high, predict hit", which is circular and useless.

`rating` (RAWG community score) is excluded because it correlates strongly with Metacritic and would not be available as a pre-release prediction feature in a real-world application.

The remaining features represent characteristics that are known at or before release — genre, platform count, ESRB rating, release timing, and tag diversity.


## Section 5 — Model Training and Evaluation

We train five classification models and compare them systematically using consistent metrics.

**Models**

| Model | Type | Why included |
|---|---|---|
| Baseline (Majority Class) | Dummy | Sanity check — all real models must beat this |
| Logistic Regression | Linear | Fast, interpretable, strong baseline for structured data |
| Random Forest | Bagging ensemble | Handles non-linear patterns, robust to outliers |
| Linear SVM | Margin-based | Finds the optimal decision boundary between classes |
| HistGradient Boosting | Boosting ensemble | Sequential trees each correcting prior errors — fast implementation |
| Stacking Classifier | Meta-learning | Combines LR and RF predictions via a meta-learner |

**Reusable function: `train_and_evaluate()`**

Trains any model and returns a consistent metrics dictionary. The same interface regardless of which model is passed in — this clean loop pattern is standard software engineering practice.


In [ ]:
def train_and_evaluate(model,
                       X_train: pd.DataFrame,
                       X_test: pd.DataFrame,
                       y_train: pd.Series,
                       y_test: pd.Series,
                       model_name: str) -> dict:
    """
    Train a model and return a consistent metrics dictionary.

    Parameters
    ----------
    model : sklearn estimator
        Any scikit-learn compatible classifier.
    X_train, X_test : pd.DataFrame
        Feature matrices for training and testing.
    y_train, y_test : pd.Series
        Target vectors for training and testing.
    model_name : str
        Label for this model in results tables and charts.

    Returns
    -------
    dict with keys: Model, Accuracy, Macro F1, ROC-AUC
    """
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    return {
        "Model":    model_name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Macro F1": round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
        "ROC-AUC":  round(roc_auc_score(y_test, proba), 4),
    }


In [ ]:
# Scale features for models sensitive to feature magnitude
# StandardScaler normalises to mean=0, std=1
# Important for Logistic Regression and SVM — not needed for tree-based models
scaler         = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=feature_names)


In [ ]:
# Define all models
# LinearSVC does not have predict_proba natively
# CalibratedClassifierCV wraps it to add probability estimates
svm_base       = LinearSVC(random_state=RANDOM_STATE, max_iter=2000, C=1.0)
svm_calibrated = CalibratedClassifierCV(svm_base, cv=3)

# Stacking: LR and RF as base learners, LR as meta-learner
stacking = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
    cv=3,
    n_jobs=-1,
)

models = {
    "Baseline (Majority Class)": DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    "Logistic Regression":       LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest":             RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "Linear SVM":                svm_calibrated,
    "HistGradient Boosting":     HistGradientBoostingClassifier(max_iter=100, random_state=RANDOM_STATE),
    "Stacking Classifier":       stacking,
}

# Models that require scaled features
SCALED_MODELS = {"Baseline (Majority Class)", "Logistic Regression", "Linear SVM"}

print("Models defined:")
for name in models:
    print(f"  {name}")


In [ ]:
# Train all models and record results
results       = []
fitted_models = {}

for name, model in models.items():
    print(f"Training: {name} ...", end="  ")
    Xtr = X_train_scaled if name in SCALED_MODELS else X_train
    Xte = X_test_scaled  if name in SCALED_MODELS else X_test

    metrics = train_and_evaluate(model, Xtr, Xte, y_train, y_test, name)
    results.append(metrics)
    fitted_models[name] = model
    print(f"F1={metrics['Macro F1']:.3f}  AUC={metrics['ROC-AUC']:.3f}")

results_df = (pd.DataFrame(results)
                .sort_values("Macro F1", ascending=False)
                .reset_index(drop=True))
print()
display(results_df)


## Section 5b — Statistical Significance of Features (Logistic Regression)

Your teacher's feedback from MP02 recommended checking the statistical significance of features, particularly for statistical models like logistic regression.

Scikit-learn's `LogisticRegression` does not provide p-values — it is optimised for prediction, not inference. To check statistical significance we use `statsmodels`, which fits the same logistic regression model but reports p-values, confidence intervals, and z-scores for each feature.

A p-value below 0.05 means we can reject the null hypothesis that the feature has no effect — i.e. the feature is statistically significant at the 95% confidence level.


In [ ]:
# Statistical significance check using statsmodels logistic regression
# We use the scaled training features for consistency with sklearn LR

# statsmodels requires a constant (intercept) term added manually
X_train_sm = sm.add_constant(X_train_scaled.values)
X_test_sm  = sm.add_constant(X_test_scaled.values)

# Fit logistic regression with statsmodels
logit_model = sm.Logit(y_train, X_train_sm)
logit_result = logit_model.fit(disp=0)  # disp=0 suppresses iteration output

# Build a clean summary table
feature_labels = ["const"] + feature_names
summary_data = {
    "Feature":   feature_labels,
    "Coef":      logit_result.params.round(4),
    "Std Err":   logit_result.bse.round(4),
    "z-score":   logit_result.tvalues.round(4),
    "p-value":   logit_result.pvalues.round(4),
    "Sig":       ["***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
                  for p in logit_result.pvalues],
}

sig_df = pd.DataFrame(summary_data)
print("Logistic Regression — Feature Statistical Significance")
print("Significance: *** p<0.001  ** p<0.01  * p<0.05")
print()
display(sig_df)


## Section 6 — Model Evaluation and Comparison

**Why Macro F1 as the primary metric?**

Our classes are close to balanced (47/53) but not perfectly even. Macro F1 treats both classes equally — it averages the F1 score for Hit and Not Hit independently, penalising any model that does well on one class but poorly on the other.

ROC-AUC is our secondary metric. It measures how well the model ranks positive examples above negative ones across all possible classification thresholds, regardless of the cutoff chosen.

**Reusable functions: `compare_models()` and `plot_confusion_matrix()`**


In [ ]:
def compare_models(results_df: pd.DataFrame,
                   baseline_name: str = "Baseline (Majority Class)") -> None:
    """
    Plot a side-by-side bar chart comparing all models on Macro F1 and ROC-AUC.

    Parameters
    ----------
    results_df : pd.DataFrame
        DataFrame with columns: Model, Accuracy, Macro F1, ROC-AUC.
    baseline_name : str
        Name of the baseline — shown as a reference line rather than a bar.
    """
    baseline_f1  = results_df[results_df["Model"] == baseline_name]["Macro F1"].values[0]
    baseline_auc = results_df[results_df["Model"] == baseline_name]["ROC-AUC"].values[0]
    plot_df      = results_df[results_df["Model"] != baseline_name].copy()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, metric, baseline_val, colour in zip(
        axes,
        ["Macro F1", "ROC-AUC"],
        [baseline_f1, baseline_auc],
        [BLUE, GREEN],
    ):
        sorted_df = plot_df.sort_values(metric)
        bars = ax.barh(sorted_df["Model"], sorted_df[metric],
                       color=colour, edgecolor="white")
        for bar, val in zip(bars, sorted_df[metric]):
            ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
                    f"{val:.3f}", va="center", fontsize=9)
        ax.axvline(baseline_val, color=CORAL, linestyle="--",
                   linewidth=1.5, label=f"Baseline ({baseline_val:.3f})")
        ax.set_xlabel(metric)
        ax.set_title(f"{metric} by model", fontweight="bold")
        ax.set_xlim(0, 1.05)
        ax.legend(fontsize=8)

    plt.suptitle("Model comparison", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("../outputs/model_comparison.png", bbox_inches="tight")
    plt.show()


In [ ]:
def plot_confusion_matrix(model,
                          X_test: pd.DataFrame,
                          y_test: pd.Series,
                          model_name: str,
                          labels: list = None) -> None:
    """
    Plot a styled confusion matrix for a fitted model.

    Parameters
    ----------
    model : fitted sklearn estimator
    X_test : pd.DataFrame
    y_test : pd.Series
    model_name : str
    labels : list, optional
        Class label names. Defaults to ['Not Hit', 'Hit'].
    """
    if labels is None:
        labels = ["Not Hit", "Hit"]

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=labels,
        cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(f"Confusion matrix — {model_name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig("../outputs/confusion_matrix.png", bbox_inches="tight")
    plt.show()


In [ ]:
# 6.1 Model comparison chart
compare_models(results_df)


**Insight:** *(Fill in after running)*


In [ ]:
# 6.2 Detailed classification report — best model
best_name  = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_Xte   = X_test_scaled if best_name in SCALED_MODELS else X_test

print(f"Best model: {best_name}")
print()
print(classification_report(y_test, best_model.predict(best_Xte),
                             target_names=["Not Hit", "Hit"]))


In [ ]:
# 6.3 Confusion matrix — best model
plot_confusion_matrix(best_model, best_Xte, y_test, best_name)


**Insight:** *(Fill in after running)*


In [ ]:
# 6.4 ROC curves — all models
fig, ax = plt.subplots(figsize=(7, 5))

for (name, model), colour in zip(fitted_models.items(), PALETTE):
    if name == "Baseline (Majority Class)":
        continue
    Xte   = X_test_scaled if name in SCALED_MODELS else X_test
    proba = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    score = sklearn_auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colour, linewidth=2,
            label=f"{name} (AUC={score:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — all models", fontweight="bold")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("../outputs/roc_curves.png", bbox_inches="tight")
plt.show()


**Insight:** *(Fill in after running)*


## Section 7 — Hyperparameter Tuning

We tune the best-performing model using GridSearchCV with stratified 3-fold cross-validation. The goal is to find the parameter combination that maximises Macro F1 on unseen data.

GridSearchCV works by:
1. Splitting the training data into 3 folds
2. Training and evaluating the model on each fold for every parameter combination
3. Returning the combination with the best average Macro F1 across all folds

**Reusable function: `run_grid_search()`**


In [ ]:
def run_grid_search(model,
                    param_grid: dict,
                    X_train: pd.DataFrame,
                    y_train: pd.Series,
                    model_name: str,
                    scoring: str = "f1_macro",
                    cv: int = 3) -> tuple:
    """
    Run GridSearchCV and return the best fitted model and parameters.

    Parameters
    ----------
    model : sklearn estimator
        Unfitted model to tune.
    param_grid : dict
        Parameter grid to search over.
    X_train, y_train : training data
    model_name : str
    scoring : str
        Metric to optimise. Default 'f1_macro'.
    cv : int
        Number of cross-validation folds.

    Returns
    -------
    best_model : fitted estimator with best parameters
    best_params : dict of best parameters found
    """
    print(f"Tuning     : {model_name}")
    print(f"Parameters : {param_grid}")
    print(f"CV folds   : {cv}  |  Scoring: {scoring}")
    print()

    gs = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring=scoring,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
        verbose=0,
    )
    gs.fit(X_train, y_train)

    print(f"Best parameters  : {gs.best_params_}")
    print(f"Best CV {scoring} : {gs.best_score_:.4f}")

    return gs.best_estimator_, gs.best_params_


In [ ]:
# Parameter grids — one per possible winning model
param_grids = {
    "Logistic Regression": {
        "C":      [0.01, 0.1, 1.0, 10.0],
        "solver": ["lbfgs", "liblinear"],
    },
    "Random Forest": {
        "n_estimators":     [100, 200],
        "max_depth":        [10, 20, None],
        "min_samples_split": [2, 5],
    },
    "HistGradient Boosting": {
        "max_iter":     [80, 150],
        "max_depth":    [4, 6, None],
        "learning_rate": [0.05, 0.1, 0.2],
    },
    "Linear SVM": {
        "C": [0.1, 1.0, 10.0],
    },
}

# Fresh model instances for tuning
fresh_models = {
    "Logistic Regression":   LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest":         RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "HistGradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    "Linear SVM":            LinearSVC(max_iter=2000, random_state=RANDOM_STATE),
}

# Select the param grid and model for whichever model won in Section 6
# Defaults to Logistic Regression if best model is not in the grid above
tune_name   = best_name if best_name in param_grids else "Logistic Regression"
tune_Xtr    = X_train_scaled if tune_name in SCALED_MODELS else X_train
tune_model  = fresh_models[tune_name]
tune_params = param_grids[tune_name]

tuned_model, best_params = run_grid_search(
    tune_model, tune_params, tune_Xtr, y_train, tune_name
)


In [ ]:
# Retrain tuned model on full training set and evaluate on test set
tune_Xtr = X_train_scaled if tune_name in SCALED_MODELS else X_train
tune_Xte = X_test_scaled  if tune_name in SCALED_MODELS else X_test

tuned_model.fit(tune_Xtr, y_train)
tuned_preds = tuned_model.predict(tune_Xte)
tuned_proba = tuned_model.predict_proba(tune_Xte)[:, 1]

tuned_f1  = f1_score(y_test, tuned_preds, average="macro", zero_division=0)
tuned_auc = roc_auc_score(y_test, tuned_proba)
tuned_acc = accuracy_score(y_test, tuned_preds)
default   = results_df[results_df["Model"] == tune_name].iloc[0]

print(f"{'Model':<35} {'Accuracy':>10} {'Macro F1':>10} {'ROC-AUC':>10}")
print("-" * 67)
print(f"{'Default ' + tune_name:<35} {default['Accuracy']:>10.4f} {default['Macro F1']:>10.4f} {default['ROC-AUC']:>10.4f}")
print(f"{'Tuned ' + tune_name:<35} {tuned_acc:>10.4f} {tuned_f1:>10.4f} {tuned_auc:>10.4f}")
print(f"{'Improvement':<35} {'':>10} {tuned_f1 - default['Macro F1']:>+10.4f} {tuned_auc - default['ROC-AUC']:>+10.4f}")


## Section 8 — Feature Importance

Feature importance answers the most interesting question of the project: what actually predicts whether a game will be a Hit?

Different model types expose feature importance differently. Tree models expose a `feature_importances_` attribute (Gini importance). Linear models expose a `coef_` attribute — the magnitude of each coefficient shows how strongly each feature pushes the prediction.

**Reusable function: `plot_feature_importance()`**

Handles both model types with the same interface.


In [ ]:
def plot_feature_importance(model,
                            feature_names: list,
                            model_name: str,
                            top_n: int = 15) -> pd.DataFrame:
    """
    Extract and plot feature importance for tree or linear models.

    Parameters
    ----------
    model : fitted sklearn estimator
        Tree model (.feature_importances_) or linear model (.coef_).
    feature_names : list
        Feature names in the same order as the model was trained on.
    model_name : str
    top_n : int
        Number of top features to display.

    Returns
    -------
    pd.DataFrame
        Feature importance scores sorted descending.
    """
    if hasattr(model, 'feature_importances_'):
        importances   = model.feature_importances_
        importance_label = "Feature Importance (Gini)"
    elif hasattr(model, 'coef_'):
        importances   = np.abs(model.coef_[0])
        importance_label = "Absolute Coefficient"
    else:
        print(f"Model type {type(model)} does not support feature importance extraction.")
        return pd.DataFrame()

    importance_df = (pd.DataFrame({
                        "Feature":    feature_names,
                        "Importance": importances,
                     })
                     .sort_values("Importance", ascending=False)
                     .head(top_n)
                     .reset_index(drop=True))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(importance_df["Feature"][::-1],
            importance_df["Importance"][::-1],
            color=BLUE, edgecolor="white")
    ax.set_xlabel(importance_label)
    ax.set_title(f"Top {top_n} features — {model_name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig("../outputs/feature_importance.png", bbox_inches="tight")
    plt.show()

    return importance_df


In [ ]:
# Plot feature importance for the tuned model
importance_df = plot_feature_importance(
    tuned_model,
    feature_names,
    f"Tuned {tune_name}",
)

print("Feature importance ranking:")
display(importance_df)


**Insight:** *(Fill in after running — which features matter most? Does this match your intuition as a gamer?)*

-
-
-


## Section 9 — Final Summary and Conclusion

### Final model comparison table


In [ ]:
# Final summary table — all models plus tuned champion
try:
    final_rows = results_df.to_dict("records")
    final_rows.append({
        "Model":    f"Tuned {tune_name} (best)",
        "Accuracy": round(tuned_acc, 4),
        "Macro F1": round(tuned_f1,  4),
        "ROC-AUC":  round(tuned_auc, 4),
    })
    final_df = (pd.DataFrame(final_rows)
                  .sort_values("Macro F1", ascending=False)
                  .reset_index(drop=True))
except NameError:
    final_df = results_df.copy()

def highlight_rows(row):
    if "best" in row["Model"]:
        return ["background-color: #d4edda"] * len(row)
    elif "Baseline" in row["Model"]:
        return ["background-color: #f8d7da"] * len(row)
    return [""] * len(row)

display(final_df.style
        .apply(highlight_rows, axis=1)
        .format({"Accuracy": "{:.4f}", "Macro F1": "{:.4f}", "ROC-AUC": "{:.4f}"}))


## Conclusion

### What we built

Using 7,141 games collected from the RAWG Video Games Database API, this project trained and evaluated five machine learning models to predict whether a game will be critically acclaimed — defined as a Metacritic score of 75 or above.

### Key findings

*(Fill in after running)*

-
-
-

### Limitations

- Metacritic coverage bias — RAWG only carries Metacritic scores for a subset of released games, likely skewed toward higher-profile titles. Smaller indie games are underrepresented.
- Developer and publisher data was unavailable through the API collection method used. Developer and publisher reputation could be significant predictors.
- Metacritic is not a perfect measure of quality. A game can score below 75 and still be deeply valued by its audience.
- The gaming industry changes rapidly. A model trained on 1995–2024 data may not generalise well to future releases.

### Future improvements

- Re-collect data with developer and publisher fields included
- Add NLP features from game descriptions as a natural extension toward the Capstone
- Build a time-aware model that accounts for how critical standards evolve by era
- Explore multi-class prediction — Flop / Average / Good / Great — for a richer output
- Deploy as a simple Streamlit app: user selects genre, platform, ESRB rating and the model returns Hit or Not Hit

Data sourced from the RAWG Video Games Database API (rawg.io) — used under free tier with attribution.
Project repository: github.com/brighthikaru/game-success-classifier
